# 1.2 单粒子散射行为模拟

**学习目标**
- 理解后向散射矩阵（Scattering Matrix）的物理意义
- 掌握粒子形状（球形、椭球形）对散射特性的影响
- 了解瑞利散射近似下散射截面与粒子尺寸的关系
- 观察粒子倾角对后向散射矩阵各分量的影响

**参考文献**：Ryzhkov & Zrnic, Chapter 1, pp. 46-85

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, FloatSlider, VBox
import ipywidgets as widgets
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 理论背景

### 散射矩阵

对于双线偏振雷达，入射场和散射场的关系通过散射矩阵描述：

$$\begin{pmatrix} E_h^s \\ E_v^s \end{pmatrix} = \frac{e^{ikR}}{R} \begin{pmatrix} S_{hh} & S_{hv} \\ S_{vh} & S_{vv} \end{pmatrix} \begin{pmatrix} E_h^i \\ E_v^i \end{pmatrix}$$

在瑞利散射近似下，对于轴对称椭球粒子：

$$S_{hh} = \frac{\pi^5}{\lambda^3} |K| D^6 \frac{\cos^2\theta \sin^2\alpha + \sin^2\theta}{\cos^2\gamma}$$

其中 $\theta$ 是粒子倾角，$\alpha$ 是入射波极化方向与粒子主轴的夹角。

In [ ]:
def rayleigh_scattering_cross_section(D, wavelength, m=1.33+0.0001j, epsilon=1.0):
    """
    计算瑞利散射截面
    D: 粒子等效直径 (m)
    wavelength: 雷达波长 (m)
    m: 复介电常数 (水的典型值 m = 1.33 + 0.0001i)
    epsilon: 粒子轴比相关参数
    """
    k = 2 * np.pi / wavelength
    K_squared = np.abs((m**2 - 1) / (m**2 + 2))**2
    sigma = (np.pi**5 / wavelength**4) * K_squared * D**6
    return sigma

def backscatter_matrix_element(SHH_amp, SVV_amp, theta, phi=0):
    """
    计算后向散射矩阵元素（简化模型）
    theta: 粒子倾角 (弧度)
    phi: 方位角 (弧度)
    """
    # 简化的瑞利散射模型
    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)
    
    # 水平和垂直偏振的后向散射幅度
    Shh = SHH_amp * cos_theta**2 + SVV_amp * sin_theta**2
    Svv = SHH_amp * sin_theta**2 + SVV_amp * cos_theta**2
    
    return Shh, Svv

def calculate_zdr(shh, svv):
    """计算差分反射率 ZDR (dB)"""
    return 10 * np.log10(np.abs(shh) / np.abs(svv) + 1e-10)

def calculate_ldr(shh, svv):
    """计算线性退极化比 LDR (dB)"""
    return 10 * np.log10(np.abs(svv) / np.abs(shh) + 1e-10)

## 2. 粒子形状与倾角对散射矩阵的影响

通过滑块调节粒子形状（轴比）和倾角，观察后向散射矩阵元素的变化。

In [ ]:
def plot_scattering_response(equiv_diameter=0.001, axis_ratio=0.8, tilt_angle=0.0, 
                             wavelength=0.10, dielectric_real=33.0):
    """绘制散射响应随倾角变化的函数"""
    
    # 复介电常数（水在X波段的典型值）
    m = dielectric_real + 0.001j
    
    # 散射截面随倾角变化
    theta_range = np.linspace(0, np.pi/2, 100)
    
    # 基准散射幅度
    sigma_base = rayleigh_scattering_cross_section(equiv_diameter, wavelength, m)
    SHH_base = np.sqrt(sigma_base) * axis_ratio
    SVV_base = np.sqrt(sigma_base)
    
    # 不同倾角下的散射幅度
    Shh_vals = []
    Svv_vals = []
    ZDR_vals = []
    LDR_vals = []
    
    for theta in theta_range:
        Shh, Svv = backscatter_matrix_element(SHH_base, SVV_base, theta)
        Shh_vals.append(np.abs(Shh))
        Svv_vals.append(np.abs(Svv))
        ZDR_vals.append(calculate_zdr(Shh, Svv))
        LDR_vals.append(calculate_ldr(Shh, Svv))
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    
    # 子图1: 散射幅度 vs 倾角
    ax1 = axes[0, 0]
    ax1.plot(np.degrees(theta_range), Shh_vals, 'b-', label='|S$_{hh}$| (水平)', linewidth=2)
    ax1.plot(np.degrees(theta_range), Svv_vals, 'r-', label='|S$_{vv}$| (垂直)', linewidth=2)
    ax1.set_xlabel('倾角 (degrees)', fontsize=10)
    ax1.set_ylabel('散射幅度', fontsize=10)
    ax1.set_title('后向散射幅度随倾角变化', fontsize=11)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 子图2: ZDR vs 倾角
    ax2 = axes[0, 1]
    ax2.plot(np.degrees(theta_range), ZDR_vals, 'g-', linewidth=2)
    ax2.set_xlabel('倾角 (degrees)', fontsize=10)
    ax2.set_ylabel('ZDR (dB)', fontsize=10)
    ax2.set_title('差分反射率 ZDR 随倾角变化', fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(-5, 10)
    
    # 子图3: 散射截面 vs 粒子尺寸
    ax3 = axes[1, 0]
    D_range = np.linspace(0.0001, 0.005, 100)  # 0.1mm to 5mm
    sigma_range = [rayleigh_scattering_cross_section(d, wavelength, m) for d in D_range]
    ax3.loglog(D_range * 1000, sigma_range, 'b-', linewidth=2)  # D in mm
    ax3.set_xlabel('粒子直径 D (mm)', fontsize=10)
    ax3.set_ylabel('散射截面 $\\sigma$ (m$^2$)', fontsize=10)
    ax3.set_title('瑞利散射截面 vs 粒子尺寸', fontsize=11)
    ax3.grid(True, alpha=0.3, which='both')
    
    # 子图4: LDR vs 倾角
    ax4 = axes[1, 1]
    ax4.plot(np.degrees(theta_range), LDR_vals, 'm-', linewidth=2)
    ax4.set_xlabel('倾角 (degrees)', fontsize=10)
    ax4.set_ylabel('LDR (dB)', fontsize=10)
    ax4.set_title('线性退极化比 LDR 随倾角变化', fontsize=11)
    ax4.grid(True, alpha=0.3)
    ax4.set_ylim(-40, 0)
    
    # 标记当前倾角位置
    current_theta = np.radians(tilt_angle)
    for ax in [ax1, ax2, ax4]:
        ax.axvline(x=tilt_angle, color='red', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()
    
    # 打印当前参数下的关键值
    Shh_curr, Svv_curr = backscatter_matrix_element(SHH_base, SVV_base, current_theta)
    print(f"\n当前参数下的计算结果:")
    print(f"  等效直径: {equiv_diameter*1000:.2f} mm")
    print(f"  轴比: {axis_ratio:.2f}")
    print(f"  倾角: {tilt_angle:.1f}°")
    print(f"  波长: {wavelength*100:.1f} cm ({'X' if wavelength < 0.05 else 'C' if wavelength < 0.1 else 'S'}波段)")
    print(f"  介电常数实部: {dielectric_real}")
    print(f"  |S$_{{hh}}$|: {np.abs(Shh_curr):.6f}")
    print(f"  |S$_{{vv}}$|: {np.abs(Svv_curr):.6f}")
    print(f"  ZDR: {calculate_zdr(Shh_curr, Svv_curr):.2f} dB")
    print(f"  LDR: {calculate_ldr(Shh_curr, Svv_curr):.2f} dB")

In [ ]:
# 交互式散射模拟
interact_scattering = interact(plot_scattering_response,
    equiv_diameter=widgets.FloatSlider(min=0.0001, max=0.005, step=0.0001, 
                                       value=0.001, description='等效直径 (m)'),
    axis_ratio=widgets.FloatSlider(min=0.5, max=1.0, step=0.05, 
                                  value=0.8, description='轴比 (b/a)'),
    tilt_angle=widgets.FloatSlider(min=0, max=90, step=1, 
                                  value=0, description='倾角 (°)'),
    wavelength=widgets.FloatSlider(min=0.03, max=0.11, step=0.01, 
                                  value=0.10, description='波长 (m)'),
    dielectric_real=widgets.FloatSlider(min=3, max=88, step=1, 
                                       value=33, description='介电常数实部')
)

## 3. 不同波段散射特性对比

不同雷达波段（X, C, S）对同一粒子的散射响应差异显著。

In [ ]:
def compare_bands():
    """对比不同波段的散射特性"""
    wavelengths = {'X-band': 0.032, 'C-band': 0.054, 'S-band': 0.107}  # 单位: m
    D_range = np.linspace(0.1, 5, 100)  # mm
    m = 33 + 0.001j  # 水的介电常数
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 子图1: 散射截面 vs 直径（对数坐标）
    ax1 = axes[0]
    colors = {'X-band': 'red', 'C-band': 'green', 'S-band': 'blue'}
    
    for band, wl in wavelengths.items():
        sigma_range = [rayleigh_scattering_cross_section(d/1000, wl, m) for d in D_range]
        ax1.loglog(D_range, sigma_range, color=colors[band], linewidth=2, label=band)
    
    ax1.set_xlabel('粒子直径 D (mm)', fontsize=11)
    ax1.set_ylabel('散射截面 $\\sigma$ (m$^2$)', fontsize=11)
    ax1.set_title('各波段散射截面对比', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3, which='both')
    
    # 子图2: 散射截面比 (相对于S波段)
    ax2 = axes[1]
    sigma_s = [rayleigh_scattering_cross_section(d/1000, wavelengths['S-band'], m) for d in D_range]
    
    for band, wl in wavelengths.items():
        sigma = [rayleigh_scattering_cross_section(d/1000, wl, m) for d in D_range]
        ratio = [s/ss for s, ss in zip(sigma, sigma_s)]
        ax2.semilogy(D_range, ratio, color=colors[band], linewidth=2, label=band)
    
    ax2.set_xlabel('粒子直径 D (mm)', fontsize=11)
    ax2.set_ylabel('$\\sigma_{band}$ / $\\sigma_{S-band}$', fontsize=11)
    ax2.set_title('相对于S波段的散射截面比', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 打印瑞利判据
    print("\n=== 瑞利散射近似适用条件 ===")
    print("瑞利近似要求: D << λ")
    for band, wl in wavelengths.items():
        D_rayleigh_max = wl / 10  # 经验规则: D < λ/10
        print(f"  {band} (λ={wl*100:.1f} cm): D < {D_rayleigh_max*1000:.1f} mm")

In [ ]:
compare_bands()

## 练习题

1. **倾角影响**：雨滴（轴比约0.6）在倾角0°（垂直下落）和45°时的ZDR值分别为多少？解释差异原因。

2. **尺寸效应**：当粒子直径从0.5mm增加到2mm时，散射截面增加多少倍？

3. **波段比较**：为什么X波段的散射截面远大于S波段？这对雷达探测有什么影响？

4. **介电常数效应**：冰的介电常数（约3.2）远小于水（33），计算相同尺寸的冰粒和水滴的散射截面比。

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 1, Springer.
- van de Hulst, H.C., 1981: *Light Scattering by Small Particles*, Dover Publications.